# Blog 5 — Idempotency in Data Engineering

## Why Rerunning Pipelines Shouldn't Duplicate Data

This notebook explains and demonstrates **idempotency** using Databricks, PySpark, Spark SQL and Delta Lake.

We will answer:

> What happens when a production data pipeline is run again with the same input?

A reliable pipeline should be safe to retry.

We will cover:

1. What idempotency means
2. Why it matters in Data Engineering
3. What can happen without it
4. Business keys and deduplication
5. Batch/run tracking and a real batch-level skip check
6. A deliberately bad append pipeline
7. Idempotent PySpark + Delta `MERGE`
8. Idempotent Spark SQL `MERGE`
9. Delete-and-reload — done correctly, and shown broken on purpose
10. Partial failure and retry, with a real failure injection
11. Validation
12. Production checklist

**Revision note.** This version fixes an issue in the earlier delete-and-reload
demonstration, where records could be partitioned by their *last-updated* date
rather than a stable business partition key. That let an updated record's old
partition (Day 1) and new partition (Day 2) both retain a copy of the same
`order_id`, producing 106,000 rows with 1,000 duplicated business keys —
while the demo only checked that row counts were stable across retries, not
that the target was actually correct. This version fixes the partitioning,
adds a duplicate-key check to every reload assertion, and includes a
deliberately-broken version alongside the fix so the failure mode is shown,
not just described. It also adds a real batch-level `SUCCESS -> SKIP` check
against the control table, an actual `dropDuplicates()` demonstration, and a
real failure injection between MERGE and the control-table update.

# 1. What Is Idempotency?

**Idempotency means that repeating the same operation produces the same final state.**

For a data pipeline:

```text
Run 1 → correct data
Run 2 → same correct data
Run 3 → same correct data
```

Without idempotency:

```text
Run 1 → 1,000 rows
Run 2 → 2,000 rows
Run 3 → 3,000 rows
```

With idempotency:

```text
Run 1 → 1,000 rows
Run 2 → 1,000 rows
Run 3 → 1,000 rows
```

The key point is not that the pipeline never fails.

The key point is that **a retry should not corrupt the final data state**.

# 2. Why Is Idempotency Important in Data Engineering?

Production pipelines are retried because of:

- Network failures
- Cluster failures
- Timeouts
- Source-system problems
- Application errors
- Manual reruns
- Orchestrator retries

A particularly important scenario is:

```text
Read source       ✓
Transform         ✓
Write target      ✓
Validation        ✗
```

The job is marked as failed.

An engineer retries it.

If the write uses blind append, data that was already written may be written again.

Therefore:

> A failed pipeline does not necessarily mean that no data was written.

Idempotency is a **pipeline reliability** concept.

# 3. What Can Happen Without Idempotency?

Suppose the source contains:

| order_id | customer | amount |
|---:|---|---:|
| 101 | Arun | 500 |
| 102 | Ravi | 750 |
| 103 | Kumar | 300 |

First run loads 3 rows.

The pipeline fails later.

The same input is processed again.

If the pipeline blindly appends:

```text
First run = 3
Retry     = 3
Total     = 6
```

Now the same business orders exist twice.

For a sales pipeline:

```text
Actual sales   = ₹1,550
Reported sales = ₹3,100
```

Duplicates can then flow downstream:

```text
Bad Silver
   ↓
Bad Gold
   ↓
Bad Dashboard
   ↓
Bad Business Decision
```

So idempotency protects not only a table, but potentially the entire downstream data flow.

# 4. Notebook Architecture

```text
Source Batch
    ↓
Batch-level check (already SUCCESS? SKIP)
    ↓
Input Validation
    ↓
Deduplicate
    ↓
Business Key
    ↓
Idempotent Write
    ↓
Target Validation
    ↓
Batch Metadata
```

For this notebook:

- `order_id` = business key
- `order_date` = stable business partition key (does **not** change when a record is later updated)
- Day 1 = initial 100,000 orders, all placed on `2026-08-18`
- Day 2 = 5,000 new orders placed on `2026-08-19`
- Day 2 = 1,000 updates to orders that were originally placed on `2026-08-18`

Correct final target:

```text
100,000 existing
+ 5,000 new
= 105,000 unique orders
```

The 1,000 updates must **update** existing rows, not create 1,000 new rows —
and, later in this notebook, must not silently duplicate across partitions
either.

# 5. Imports and Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

print("Spark:", spark.version)

BASE = "blog5"

TARGET_BAD = f"{BASE}_bad_append"
TARGET_IDEMPOTENT = f"{BASE}_idempotent"
TARGET_SQL = f"{BASE}_sql"
TARGET_RELOAD = f"{BASE}_delete_reload"
TARGET_RELOAD_BROKEN = f"{BASE}_delete_reload_broken"
CONTROL = f"{BASE}_control"

INITIAL_ORDERS = 100_000
NEW_ORDERS = 5_000
UPDATED_ORDERS = 1_000

DAY1_TS = "2026-08-18 10:00:00"
DAY2_TS = "2026-08-19 10:00:00"

DAY1_DATE = "2026-08-18"
DAY2_DATE = "2026-08-19"

BATCH_DAY1 = "BATCH_2026_08_18"
BATCH_DAY2 = "BATCH_2026_08_19"

# A separate batch id for the partial-failure demo in Section 30. It must be
# a batch that has never been recorded as SUCCESS yet, so the failure/retry
# story starts from a clean slate rather than colliding with BATCH_DAY2,
# which Section 22 already marks SUCCESS for the batch-level SKIP demo.
BATCH_DAY2_RETRY_DEMO = "BATCH_2026_08_19_RETRY_DEMO"


Spark: 4.1.0


# 6. Clean Previous Tutorial State

In [0]:
for table_name in [
    TARGET_BAD,
    TARGET_IDEMPOTENT,
    TARGET_SQL,
    TARGET_RELOAD,
    TARGET_RELOAD_BROKEN,
    CONTROL
]:
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

print("Clean.")


Clean.


# 7. Create Day 1 — Initial Orders

Each order gets two time columns with different meanings:

- `updated_at` — when the row was last written. This changes on every update.
- `order_date` — when the order was originally placed. This is a **stable
  business fact** that must not change just because the order's status
  changes later. We will use `order_date` as the partition key in the
  delete-and-reload section, specifically because it doesn't drift.

In [0]:
orders_day1 = (
    spark.range(1, INITIAL_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % 10_000) + 1)
    .withColumn("amount", F.round(F.lit(100.0) + (F.col("order_id") % 500) * 10.0, 2))
    .withColumn("status", F.lit("COMPLETED"))
    .withColumn("updated_at", F.to_timestamp(F.lit(DAY1_TS)))
    .withColumn("order_date", F.to_date(F.lit(DAY1_DATE)))
    .withColumn("batch_id", F.lit(BATCH_DAY1))
)

print("Rows:", orders_day1.count())
display(orders_day1.limit(10))

assert orders_day1.count() == INITIAL_ORDERS
assert orders_day1.select("order_id").distinct().count() == INITIAL_ORDERS
print("PASS — Day 1 validation.")


Rows: 100000


order_id,customer_id,amount,status,updated_at,order_date,batch_id
1,18,110.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
2,35,120.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
3,52,130.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
4,69,140.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
5,86,150.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
6,103,160.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
7,120,170.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
8,137,180.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
9,154,190.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18
10,171,200.0,COMPLETED,2026-08-18T10:00:00.000Z,2026-08-18,BATCH_2026_08_18


PASS — Day 1 validation.


# 8. Create Day 2 — New + Updated Records

`new_orders` were placed today, so `order_date = 2026-08-19`.

`updated_orders` are **existing Day 1 orders** whose status changed today —
their `updated_at` moves to Day 2, but their `order_date` stays `2026-08-18`,
because that is when they were actually placed. This detail is exactly what
the delete-and-reload section depends on getting right.

In [0]:
new_orders = (
    spark.range(INITIAL_ORDERS + 1, INITIAL_ORDERS + NEW_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % 10_000) + 1)
    .withColumn("amount", F.round(F.lit(100.0) + (F.col("order_id") % 500) * 10.0, 2))
    .withColumn("status", F.lit("COMPLETED"))
    .withColumn("updated_at", F.to_timestamp(F.lit(DAY2_TS)))
    .withColumn("order_date", F.to_date(F.lit(DAY2_DATE)))
    .withColumn("batch_id", F.lit(BATCH_DAY2))
)

updated_orders = (
    spark.range(1, UPDATED_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", ((F.col("order_id") * 17) % 10_000) + 1)
    .withColumn("amount", F.round(F.lit(999.0) + (F.col("order_id") % 100) * 5.0, 2))
    .withColumn("status", F.lit("REFUNDED"))
    .withColumn("updated_at", F.to_timestamp(F.lit(DAY2_TS)))
    .withColumn("order_date", F.to_date(F.lit(DAY1_DATE)))   # FIX: original placement date, not today
    .withColumn("batch_id", F.lit(BATCH_DAY2))
)

day2_batch = new_orders.unionByName(updated_orders)

print("New:", new_orders.count())
print("Updated:", updated_orders.count())
print("Total Day 2 input:", day2_batch.count())

assert day2_batch.count() == NEW_ORDERS + UPDATED_ORDERS


New: 5000
Updated: 1000
Total Day 2 input: 6000


# 9. The Wrong Approach — Blind Append

In [0]:
orders_day1.write.format("delta").mode("overwrite").saveAsTable(TARGET_BAD)

day2_batch.write.format("delta").mode("append").saveAsTable(TARGET_BAD)

bad = spark.table(TARGET_BAD)

bad_count = bad.count()
duplicate_keys = (
    bad.groupBy("order_id")
       .count()
       .filter(F.col("count") > 1)
       .count()
)

print("Rows after append:", bad_count)
print("Duplicate business keys:", duplicate_keys)

assert bad_count == 106_000
assert duplicate_keys == UPDATED_ORDERS

print("PASS — append demonstrated the idempotency problem.")


Rows after append: 106000
Duplicate business keys: 1000
PASS — append demonstrated the idempotency problem.


# 10. Why Did Append Create Bad Data?

The target started with 100,000 orders.

Day 2 contains:

```text
5,000 new orders
1,000 existing orders
```

Blind append does:

```text
100,000 existing
+ 5,000 new
+ 1,000 existing again
= 106,000 rows
```

But the correct target is:

```text
100,000 existing
+ 5,000 new
= 105,000 unique orders
```

The 1,000 existing orders should be **updated**, not inserted again.

# 11. Business Key

A **business key** identifies a business record.

Here:

```text
order_id
```

is the business key.

It lets the pipeline ask:

> Does this order already exist?

Conceptually:

```text
Incoming order
      ↓
Does order_id exist?
   ↙          ↘
 YES           NO
  ↓             ↓
UPDATE        INSERT
```

Without a reliable business key, idempotent record-level processing becomes much harder.

# 12. Source Deduplication

In [0]:
source_duplicates = (
    day2_batch
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate keys in source:", source_duplicates)

assert source_duplicates == 0
print("PASS — incoming batch has unique business keys.")


Duplicate keys in source: 0
PASS — incoming batch has unique business keys.


# 13. Deduplication Is Not the Same as Idempotency

`dropDuplicates()` can remove duplicates **inside the incoming DataFrame**.

It does not automatically protect against records already present in the target.

For example:

```text
Target:
101
102

Incoming:
101
102
103
```

The incoming DataFrame itself has no duplicates.

But appending it still creates duplicate 101 and 102.

Therefore:

```text
Deduplication
     +
Idempotent target write
```

are separate concerns.

# 14. `dropDuplicates()` in Practice

The claim in the previous section is worth proving, not just stating.

We build a small source with two conflicting versions of `order_id = 101`
(different amount, different status, different `updated_at`), and compare:

- `dropDuplicates(["order_id"])` — removes duplicates, but Spark does **not**
  guarantee which of the conflicting rows survives.
- The window-function approach from the next section — deterministically
  keeps the row with the latest `updated_at`, i.e. an actual business rule.

In [0]:
conflicting_source = spark.createDataFrame(
    [
        (101, 1, 500.0, "COMPLETED", "2026-08-19 09:00:00"),
        (101, 1, 550.0, "REFUNDED",  "2026-08-19 11:00:00"),
        (102, 2, 750.0, "COMPLETED", "2026-08-19 10:00:00"),
    ],
    "order_id LONG, customer_id LONG, amount DOUBLE, status STRING, updated_at STRING"
).withColumn("updated_at", F.to_timestamp("updated_at"))

naive_dedup = conflicting_source.dropDuplicates(["order_id"])

print("Rows before dropDuplicates:", conflicting_source.count())
print("Rows after dropDuplicates :", naive_dedup.count())
display(naive_dedup.orderBy("order_id"))

assert naive_dedup.count() == 2

# dropDuplicates() removed a duplicate order_id, which looks like progress.
# But nothing here guarantees the REFUNDED / 550.0 (latest) row was the one kept.
# Run this cell a few times or change partition layout and the surviving row
# for order_id 101 is not guaranteed to be the same one.
kept_amount_for_101 = (
    naive_dedup.filter(F.col("order_id") == 101).first()["amount"]
)
print("Row kept for order_id 101 by dropDuplicates():", kept_amount_for_101)
print("This may or may not be the business-correct (latest) version — that is the problem.")


Rows before dropDuplicates: 3
Rows after dropDuplicates : 2


order_id,customer_id,amount,status,updated_at
101,1,500.0,COMPLETED,2026-08-19T09:00:00.000Z
102,2,750.0,COMPLETED,2026-08-19T10:00:00.000Z


Row kept for order_id 101 by dropDuplicates(): 500.0
This may or may not be the business-correct (latest) version — that is the problem.


# 15. Negative Test — Deduplicating with an Explicit Business Rule

In [0]:
bad_source = spark.createDataFrame(
    [
        (101, 1, 500.0, "COMPLETED", "2026-08-19 10:00:00"),
        (101, 1, 550.0, "REFUNDED", "2026-08-19 11:00:00"),
        (102, 2, 750.0, "COMPLETED", "2026-08-19 10:00:00"),
    ],
    "order_id LONG, customer_id LONG, amount DOUBLE, status STRING, updated_at STRING"
).withColumn("updated_at", F.to_timestamp("updated_at"))

display(bad_source)

w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

deduped = (
    bad_source
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(deduped)

assert deduped.count() == 2
assert deduped.filter(F.col("order_id") == 101).first()["amount"] == 550.0
print("PASS — latest record retained using an explicit, deterministic business rule.")
print("Unlike dropDuplicates(), this is guaranteed to keep the REFUNDED / 550.0 row every time.")


order_id,customer_id,amount,status,updated_at
101,1,500.0,COMPLETED,2026-08-19T10:00:00.000Z
101,1,550.0,REFUNDED,2026-08-19T11:00:00.000Z
102,2,750.0,COMPLETED,2026-08-19T10:00:00.000Z


order_id,customer_id,amount,status,updated_at
101,1,550.0,REFUNDED,2026-08-19T11:00:00.000Z
102,2,750.0,COMPLETED,2026-08-19T10:00:00.000Z


PASS — latest record retained using an explicit, deterministic business rule.
Unlike dropDuplicates(), this is guaranteed to keep the REFUNDED / 550.0 row every time.


# 16. Batch IDs and Run Tracking

In [0]:
control_schema = """
pipeline_name STRING,
batch_id STRING,
status STRING,
record_count LONG,
processed_at TIMESTAMP
"""

control_day1 = spark.createDataFrame(
    [("blog5_orders", BATCH_DAY1, "SUCCESS", INITIAL_ORDERS, datetime.now())],
    control_schema
)

control_day1.write.format("delta").mode("overwrite").saveAsTable(CONTROL)

display(spark.table(CONTROL))


pipeline_name,batch_id,status,record_count,processed_at
blog5_orders,BATCH_2026_08_18,SUCCESS,100000,2026-08-19T09:34:38.175Z


A batch ID tells us which logical group of records is being processed.

For example:

```text
BATCH_2026_08_19
```

This helps with:

- Retry handling
- Auditing
- Monitoring
- Recovery
- Operational debugging

Batch-level tracking and record-level business keys work **together**, not
as substitutes for each other. Section 22 puts the batch ID to actual use.

# 17. PySpark + Delta MERGE — Idempotent Write

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TARGET_IDEMPOTENT}")

orders_day1.write.format("delta").mode("overwrite").saveAsTable(TARGET_IDEMPOTENT)

target = DeltaTable.forName(spark, TARGET_IDEMPOTENT)

(
    target.alias("t")
    .merge(
        day2_batch.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

result = spark.table(TARGET_IDEMPOTENT)

print("Rows after MERGE:", result.count())


Rows after MERGE: 105000


The `MERGE` logic is:

```text
If order_id already exists
    → UPDATE

If order_id does not exist
    → INSERT
```

This is the key difference from blind append.

# 18. Validate the First MERGE

In [0]:
result = spark.table(TARGET_IDEMPOTENT)

row_count = result.count()

duplicate_keys = (
    result.groupBy("order_id")
          .count()
          .filter(F.col("count") > 1)
          .count()
)

updated_count = (
    result.filter(
        (F.col("order_id") <= UPDATED_ORDERS) &
        (F.col("status") == "REFUNDED")
    ).count()
)

print("Rows:", row_count)
print("Duplicate keys:", duplicate_keys)
print("Updated records:", updated_count)

assert row_count == 105_000
assert duplicate_keys == 0
assert updated_count == UPDATED_ORDERS

print("PASS — MERGE inserted new rows and updated existing rows.")


Rows: 105000
Duplicate keys: 0
Updated records: 1000
PASS — MERGE inserted new rows and updated existing rows.


# 19. The Most Important Test — Retry the Same Batch

In [0]:
target = DeltaTable.forName(spark, TARGET_IDEMPOTENT)

# Exact same input, exact same business-key match.
target.alias("t").merge(
    day2_batch.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

retry_count = spark.table(TARGET_IDEMPOTENT).count()

print("Rows after retry:", retry_count)

assert retry_count == 105_000
print("PASS — retry did not create duplicates.")


Rows after retry: 105000
PASS — retry did not create duplicates.


# 20. Run It Three Times

In [0]:
counts = []

for run_number in range(1, 4):
    target = DeltaTable.forName(spark, TARGET_IDEMPOTENT)

    target.alias("t").merge(
        day2_batch.alias("s"),
        "t.order_id = s.order_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    current = spark.table(TARGET_IDEMPOTENT).count()
    counts.append(current)
    print(f"Execution {run_number}: {current:,} rows")

assert counts == [105_000, 105_000, 105_000]
print("PASS — repeated execution produces the same final state.")


Execution 1: 105,000 rows
Execution 2: 105,000 rows
Execution 3: 105,000 rows
PASS — repeated execution produces the same final state.


# 21. PySpark Validation Suite

In [0]:
final_target = spark.table(TARGET_IDEMPOTENT)

checks = {
    "expected_row_count": final_target.count() == 105_000,
    "no_null_order_id": final_target.filter(F.col("order_id").isNull()).count() == 0,
    "no_duplicate_order_id": (
        final_target.groupBy("order_id")
                    .count()
                    .filter(F.col("count") > 1)
                    .count() == 0
    ),
    "all_updates_applied": (
        final_target.filter(
            (F.col("order_id") <= UPDATED_ORDERS) &
            (F.col("status") == "REFUNDED")
        ).count() == UPDATED_ORDERS
    )
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

assert all(checks.values())
print("ALL PYSPARK VALIDATION CHECKS PASSED.")


PASS — expected_row_count
PASS — no_null_order_id
PASS — no_duplicate_order_id
PASS — all_updates_applied
ALL PYSPARK VALIDATION CHECKS PASSED.


# 22. Record Successful Batch Metadata

Now that the MERGE has run and been validated, we record the batch as
`SUCCESS`. This has to happen **after** the write and validation, not before —
and recording it here (rather than at the very end of the notebook) is what
lets the next section demonstrate a real batch-level skip check.

In [0]:
successful_batch = spark.createDataFrame(
    [
        (
            "blog5_orders",
            BATCH_DAY2,
            "SUCCESS",
            day2_batch.count(),
            datetime.now()
        )
    ],
    control_schema
)

successful_batch.write.format("delta").mode("append").saveAsTable(CONTROL)

display(spark.table(CONTROL).orderBy(F.col("processed_at").desc()))


pipeline_name,batch_id,status,record_count,processed_at
blog5_orders,BATCH_2026_08_19,SUCCESS,6000,2026-08-19T09:35:14.635Z
blog5_orders,BATCH_2026_08_18,SUCCESS,100000,2026-08-19T09:34:38.175Z


# 23. Batch-Level Idempotency Check

Record-level `MERGE` idempotency (sections 17–20) protects the data even if
the same batch is reprocessed. But it still does real work — reading the
source, deduplicating, and running a MERGE — every time it is triggered.

A batch-level check is a cheaper first line of defense: if the control table
already shows this exact `batch_id` as `SUCCESS`, skip the batch entirely
before touching the target table.

```text
Batch already SUCCESS → SKIP
Batch not SUCCESS     → PROCESS
```

This is **not** a replacement for MERGE-based idempotency — it is a
complementary, cheaper gate that sits in front of it. If the control table
itself was never updated (for example because of the failure in Section 30),
this check will not catch it, and MERGE idempotency is what protects the
target in that case.

In [0]:
def check_batch_status(pipeline_name, batch_id):
    already_succeeded = (
        spark.table(CONTROL)
        .filter(F.col("pipeline_name") == pipeline_name)
        .filter(F.col("batch_id") == batch_id)
        .filter(F.col("status") == "SUCCESS")
        .count()
        > 0
    )
    return "SKIP" if already_succeeded else "PROCESS"


# BATCH_DAY2 was just recorded as SUCCESS in Section 22.
decision_existing_batch = check_batch_status("blog5_orders", BATCH_DAY2)
print(f"Decision for {BATCH_DAY2}: {decision_existing_batch}")
assert decision_existing_batch == "SKIP"

target_count_before = spark.table(TARGET_IDEMPOTENT).count()

if decision_existing_batch == "SKIP":
    print("Batch already SUCCESS — skipping reprocessing. Target is not touched.")
else:
    # A real pipeline would run the MERGE here.
    pass

target_count_after = spark.table(TARGET_IDEMPOTENT).count()

assert target_count_before == target_count_after == 105_000
print("PASS — target row count unchanged; MERGE was never invoked for a SKIPped batch.")

# Contrast: a batch that has not run yet should be PROCESSed.
new_batch_id = "BATCH_2026_08_20"
decision_new_batch = check_batch_status("blog5_orders", new_batch_id)
print(f"Decision for {new_batch_id}: {decision_new_batch}")
assert decision_new_batch == "PROCESS"


Decision for BATCH_2026_08_19: SKIP
Batch already SUCCESS — skipping reprocessing. Target is not touched.
PASS — target row count unchanged; MERGE was never invoked for a SKIPped batch.
Decision for BATCH_2026_08_20: PROCESS


# 24. Spark SQL Implementation

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TARGET_SQL}")

orders_day1.write.format("delta").mode("overwrite").saveAsTable(TARGET_SQL)

day2_batch.createOrReplaceTempView("blog5_day2_batch")


## SQL MERGE

```sql
MERGE INTO target AS t
USING source AS s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
```

The business key determines whether the record is updated or inserted.

In [0]:
spark.sql(f"""
MERGE INTO {TARGET_SQL} AS t
USING blog5_day2_batch AS s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
""")

sql_count = spark.table(TARGET_SQL).count()

sql_duplicates = (
    spark.table(TARGET_SQL)
         .groupBy("order_id")
         .count()
         .filter(F.col("count") > 1)
         .count()
)

print("SQL MERGE rows:", sql_count)
print("SQL duplicate keys:", sql_duplicates)

assert sql_count == 105_000
assert sql_duplicates == 0
print("PASS — SQL MERGE validation.")


SQL MERGE rows: 105000
SQL duplicate keys: 0
PASS — SQL MERGE validation.


# 25. SQL Retry Validation

In [0]:
spark.sql(f"""
MERGE INTO {TARGET_SQL} AS t
USING blog5_day2_batch AS s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
""")

retry_sql_count = spark.table(TARGET_SQL).count()

print("SQL rows after retry:", retry_sql_count)

assert retry_sql_count == 105_000
print("PASS — SQL MERGE is idempotent for this workload.")


SQL rows after retry: 105000
PASS — SQL MERGE is idempotent for this workload.


# 26. Delete-and-Reload — Another Idempotent Pattern

If the pipeline processes a **complete partition**, another practical pattern is:

```text
Delete existing partition
        ↓
Load the complete, current state of that partition
```

For example:

```sql
DELETE FROM sales
WHERE order_date = '2026-08-18';
```

Then reload every row that currently belongs to that partition.

This is useful when:

- The partition is rebuilt completely
- The volume is manageable
- Full partition replacement is simpler than row-level upsert

**The pattern only works cleanly if the partition key fully owns its business
keys** — i.e. a given `order_id` must always belong to the same partition,
even after an update. `order_date` (the date the order was originally
placed) satisfies this. `updated_at` (the date the row was last written)
does **not** — an update can move a row into a different partition than the
one holding its old copy, and a partial delete-and-reload will then leave
the old copy behind. Section 28 demonstrates this failure directly.

# 27. Delete-and-Reload Done Correctly — Partition Owns Its Keys

`order_date` is stable, so partition `2026-08-18` fully owns order_ids
1–100,000 for their entire lifetime, including the 1,000 that get refunded
on Day 2. To rebuild that partition correctly, we reload the **complete
current state** of the partition — not just the incremental delta — because
delete-and-reload always replaces the whole partition, not merges into it.

In [0]:
reload_day1 = orders_day1.withColumn("sales_partition", F.col("order_date"))
reload_new = new_orders.withColumn("sales_partition", F.col("order_date"))
reload_updated = updated_orders.withColumn("sales_partition", F.col("order_date"))

# Full current state of the 2026-08-18 partition: original Day-1 rows,
# with the 1,000 refunded orders replaced by their current (updated) version.
current_0818_partition = (
    reload_day1
    .join(reload_updated.select("order_id"), on="order_id", how="left_anti")
    .unionByName(reload_updated)
)

assert current_0818_partition.count() == INITIAL_ORDERS
assert current_0818_partition.select("order_id").distinct().count() == INITIAL_ORDERS

spark.sql(f"DROP TABLE IF EXISTS {TARGET_RELOAD}")
current_0818_partition.write.format("delta").mode("overwrite").saveAsTable(TARGET_RELOAD)
reload_new.write.format("delta").mode("append").saveAsTable(TARGET_RELOAD)

first_count = spark.table(TARGET_RELOAD).count()
first_dupes = (
    spark.table(TARGET_RELOAD).groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

print("After first load — rows:", first_count, "| duplicate keys:", first_dupes)
assert first_count == 105_000
assert first_dupes == 0

# Retry: delete and reload the SAME partitions with the SAME complete data.
spark.sql(f"DELETE FROM {TARGET_RELOAD} WHERE sales_partition = DATE('{DAY1_DATE}')")
current_0818_partition.write.format("delta").mode("append").saveAsTable(TARGET_RELOAD)

spark.sql(f"DELETE FROM {TARGET_RELOAD} WHERE sales_partition = DATE('{DAY2_DATE}')")
reload_new.write.format("delta").mode("append").saveAsTable(TARGET_RELOAD)

second_count = spark.table(TARGET_RELOAD).count()
second_dupes = (
    spark.table(TARGET_RELOAD).groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

print("After retry — rows:", second_count, "| duplicate keys:", second_dupes)

assert second_count == first_count == 105_000
assert second_dupes == 0

print("PASS — delete-and-reload is idempotent AND correct: stable row count, zero duplicate keys.")


After first load — rows: 105000 | duplicate keys: 0
After retry — rows: 105000 | duplicate keys: 0
PASS — delete-and-reload is idempotent AND correct: stable row count, zero duplicate keys.


# 28. Why Cross-Partition Updates Break This Pattern

Now we deliberately repeat the mistake this notebook originally shipped
with: partitioning by `updated_at` instead of `order_date`. Because an
update moves `updated_at` forward, the refunded orders land in the
`2026-08-19` partition while their original rows stay in `2026-08-18`. A
delete-and-reload of only the `2026-08-19` partition never removes the stale
`2026-08-18` copies.

In [0]:
broken_day1 = orders_day1.withColumn("sales_partition", F.to_date("updated_at"))
broken_day2 = day2_batch.withColumn("sales_partition", F.to_date("updated_at"))

spark.sql(f"DROP TABLE IF EXISTS {TARGET_RELOAD_BROKEN}")
broken_day1.write.format("delta").mode("overwrite").saveAsTable(TARGET_RELOAD_BROKEN)

# "Reload" the 2026-08-19 partition only, the way a naive implementation would.
spark.sql(f"DELETE FROM {TARGET_RELOAD_BROKEN} WHERE sales_partition = DATE('{DAY2_DATE}')")
broken_day2.write.format("delta").mode("append").saveAsTable(TARGET_RELOAD_BROKEN)

broken_count = spark.table(TARGET_RELOAD_BROKEN).count()
broken_dupes = (
    spark.table(TARGET_RELOAD_BROKEN).groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

print("Broken pattern — rows:", broken_count, "| duplicate keys:", broken_dupes)

# This is the bug: the target has grown and now contains real duplicate
# business keys, even though every individual reload step "succeeded".
assert broken_count == 106_000
assert broken_dupes == UPDATED_ORDERS

print("CONFIRMED BROKEN — partitioning by updated_at reintroduced the exact")
print("duplicate-key problem the blind-append demo in Section 9 was built to show.")


Broken pattern — rows: 106000 | duplicate keys: 1000
CONFIRMED BROKEN — partitioning by updated_at reintroduced the exact
duplicate-key problem the blind-append demo in Section 9 was built to show.


### Lesson

Delete-and-reload is only as correct as its partition key. If the partition
key can change for a business record (as `updated_at` does on every update),
a partial reload of "the changed partition" silently leaves stale copies
behind in other partitions — and, unlike blind append, this failure mode is
easy to miss because each individual step looks like it succeeded.

Use a stable business attribute as the partition key (`order_date`,
placement date, event date), never a mutable audit column
(`updated_at`, `processed_at`, `ingested_at`).

# 29. When to Prefer `MERGE` Over Delete-and-Reload

| Situation | Prefer |
|---|---|
| Partition key is stable and owns its business keys | Delete-and-reload is fine |
| Records can move between partitions on update | `MERGE` |
| Only a small fraction of a large partition changes | `MERGE` (avoids rewriting the whole partition) |
| A partition needs a full, clean rebuild | Delete-and-reload |
| The partition key itself is derived from a mutable column | `MERGE`, or fix the partition key first |

`MERGE` is forgiving of exactly the mistake Section 28 makes: because it
matches on `order_id` directly, it doesn't matter which partition a record
"used to" belong to. Delete-and-reload has no such protection — it trusts
the partition boundary completely.

# 30. Partial Failure and Safe Retry — Real Failure Injection

Previously this section only described the scenario in prose. Here we
actually reproduce it: the `MERGE` commits successfully, then the
control-table update raises a real exception before it can run.

This demo needs a batch that has **not already been marked `SUCCESS`** —
`BATCH_DAY2` doesn't qualify any more, because Section 22 already recorded
it as `SUCCESS` (that's what let Section 23 demonstrate the `SKIP` path). So
we use `BATCH_DAY2_RETRY_DEMO`, a separate batch id representing a fresh
pipeline run over the same Day-2 data, starting from a genuinely clean
control-table state.

In [0]:
FAILURE_TARGET = TARGET_IDEMPOTENT  # reuse the already-idempotent target
FAILURE_BATCH_ID = BATCH_DAY2_RETRY_DEMO

decision_before_failure = check_batch_status("blog5_orders", FAILURE_BATCH_ID)
print(f"Decision for {FAILURE_BATCH_ID} before any run: {decision_before_failure}")
assert decision_before_failure == "PROCESS"

target = DeltaTable.forName(spark, FAILURE_TARGET)

# Step 1: the MERGE itself succeeds and commits.
# (day2_batch's business keys are already present in FAILURE_TARGET from
#  earlier sections, so this MERGE is itself idempotent — it re-applies the
#  same values rather than growing the table. That mirrors a real pipeline
#  retrying against a target that already has this batch's effects applied.)
target.alias("t").merge(
    day2_batch.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

merge_committed_count = spark.table(FAILURE_TARGET).count()
assert merge_committed_count == 105_000
print("MERGE committed. Target rows:", merge_committed_count)

# Step 2: the control-table update fails for real, before it can run.
state_update_failed = False
try:
    raise RuntimeError("SIMULATED FAILURE: control-table update failed after committed MERGE")
    # unreachable — a real control-table UPDATE/append would go here
except RuntimeError as e:
    state_update_failed = True
    print("Caught expected failure:", e)

assert state_update_failed

# The control table was never written for this batch id — confirm it still shows PROCESS.
decision_after_failure = check_batch_status("blog5_orders", FAILURE_BATCH_ID)
print(f"Decision for {FAILURE_BATCH_ID} after the failed step: {decision_after_failure}")
assert decision_after_failure == "PROCESS"
print("PASS — MERGE committed, but the batch is still unmarked and recoverable.")


Decision for BATCH_2026_08_19_RETRY_DEMO before any run: PROCESS
MERGE committed. Target rows: 105000
Caught expected failure: SIMULATED FAILURE: control-table update failed after committed MERGE
Decision for BATCH_2026_08_19_RETRY_DEMO after the failed step: PROCESS
PASS — MERGE committed, but the batch is still unmarked and recoverable.


## Retry the Same Batch After the Failure

The orchestrator sees the batch never reached `SUCCESS` and retries it. The
batch-level check correctly says `PROCESS` (not `SKIP`), because the control
table was never updated — this is exactly the case that check cannot catch
on its own, and why record-level `MERGE` idempotency still has to hold even
when the cheaper batch-level gate doesn't fire.

In [0]:
retry_decision = check_batch_status("blog5_orders", FAILURE_BATCH_ID)
print("Batch decision on retry:", retry_decision)
assert retry_decision == "PROCESS"

target = DeltaTable.forName(spark, FAILURE_TARGET)
target.alias("t").merge(
    day2_batch.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

recovery_count = spark.table(FAILURE_TARGET).count()
recovery_dupes = (
    spark.table(FAILURE_TARGET).groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

print("Rows after recovery retry:", recovery_count)
print("Duplicate keys after recovery retry:", recovery_dupes)

assert recovery_count == 105_000
assert recovery_dupes == 0

# This time, actually record success — for FAILURE_BATCH_ID, not BATCH_DAY2.
recovered_batch = spark.createDataFrame(
    [("blog5_orders", FAILURE_BATCH_ID, "SUCCESS", day2_batch.count(), datetime.now())],
    control_schema
)
recovered_batch.write.format("delta").mode("append").saveAsTable(CONTROL)

decision_after_recovery = check_batch_status("blog5_orders", FAILURE_BATCH_ID)
assert decision_after_recovery == "SKIP"

print("PASS — partial-failure retry recovered with correct row count and zero duplicates.")
print(f"PASS — {FAILURE_BATCH_ID} now correctly shows SKIP on any further retry.")


Batch decision on retry: PROCESS
Rows after recovery retry: 105000
Duplicate keys after recovery retry: 0
PASS — partial-failure retry recovered with correct row count and zero duplicates.
PASS — BATCH_2026_08_19_RETRY_DEMO now correctly shows SKIP on any further retry.


# 31. Final End-to-End Validation

In [0]:
final_target = spark.table(TARGET_IDEMPOTENT)

final_row_count = final_target.count()

final_duplicate_keys = (
    final_target.groupBy("order_id")
                .count()
                .filter(F.col("count") > 1)
                .count()
)

final_null_keys = final_target.filter(F.col("order_id").isNull()).count()

final_updates = final_target.filter(
    (F.col("order_id") <= UPDATED_ORDERS) &
    (F.col("status") == "REFUNDED")
).count()

reload_final = spark.table(TARGET_RELOAD)
reload_final_count = reload_final.count()
reload_final_dupes = (
    reload_final.groupBy("order_id").count().filter(F.col("count") > 1).count()
)

print("Idempotent target — rows:", final_row_count)
print("Idempotent target — duplicate keys:", final_duplicate_keys)
print("Idempotent target — null keys:", final_null_keys)
print("Idempotent target — updated records:", final_updates)
print("Delete-and-reload target — rows:", reload_final_count)
print("Delete-and-reload target — duplicate keys:", reload_final_dupes)

assert final_row_count == 105_000
assert final_duplicate_keys == 0
assert final_null_keys == 0
assert final_updates == UPDATED_ORDERS
assert reload_final_count == 105_000
assert reload_final_dupes == 0

print("ALL END-TO-END VALIDATIONS PASSED — including the delete-and-reload target.")


Idempotent target — rows: 105000
Idempotent target — duplicate keys: 0
Idempotent target — null keys: 0
Idempotent target — updated records: 1000
Delete-and-reload target — rows: 105000
Delete-and-reload target — duplicate keys: 0
ALL END-TO-END VALIDATIONS PASSED — including the delete-and-reload target.


# 32. Idempotency Test Matrix

| Test | Expected result |
|---|---|
| Run batch once | Correct final state |
| Run same batch twice | Same final state |
| Run same batch three times | Same final state |
| Batch already marked SUCCESS | Batch-level check returns SKIP |
| New batch, never processed | Batch-level check returns PROCESS |
| Retry after partial failure (control update failed) | No duplicates, correct row count |
| Duplicate input records | Deduplicate using an explicit business rule, not `dropDuplicates()` alone |
| Existing order arrives again | Update existing row |
| New order arrives | Insert new row |
| Null business key | Fail validation |
| Duplicate target key | Fail validation |
| Delete-and-reload with a stable partition key | Same row count and zero duplicates on retry |
| Delete-and-reload with a mutable partition key | **Fails** — duplicate keys across partitions |

A production pipeline should test **failure and retry behavior**, not only the happy path —
and it should test that a delete-and-reload's partition key is actually safe to use.

# 33. Common Mistakes

### 1. Blind append

```python
df.write.mode("append")
```

Can create duplicates when records are reprocessed.

### 2. Thinking `dropDuplicates()` solves everything

It only handles duplicates in the incoming DataFrame, and does not
guarantee which conflicting row survives.

### 3. No business key

The pipeline cannot reliably identify an existing business record.

### 4. Partitioning by a mutable column

If the partition key can change when a record is updated (for example
`updated_at`), delete-and-reload can leave stale copies behind in the
record's old partition. Use a stable business attribute instead.

### 5. Ignoring partial failures

A job may fail after writing data.

### 6. Marking success too early

Record `SUCCESS` only after write and validation.

### 7. Never testing retries

A pipeline that works once is not automatically idempotent.

### 8. No batch-level check

Without checking the control table before processing, a pipeline redoes
real work for a batch that already succeeded — record-level `MERGE`
idempotency will keep the *data* correct, but the pipeline still wastes
compute reprocessing something it didn't need to.

# 34. Data Engineer Decision Guide

| Situation | Common approach |
|---|---|
| Immutable new events | Append can be appropriate |
| New + changed records | MERGE / upsert |
| Complete partition replacement, stable partition key | Delete + reload |
| Complete partition replacement, records can change partition | MERGE, or fix the partition key |
| Duplicate source rows | Deduplicate + explicit business rule |
| Same batch can arrive again | Batch ID + batch-level SUCCESS check + idempotent write |
| Partial failures possible | Design for safe retry (MERGE is naturally retry-safe) |
| Small number of changed records | MERGE |
| Full partition rebuild | Delete + reload |

There is no single idempotency technique for every workload.

The correct approach depends on the **business meaning of the data**.

# 35. Final Production Pattern

```text
                SOURCE
                   ↓
             Identify Batch
                   ↓
     Batch-level check (SUCCESS? → SKIP)
                   ↓
          Incremental Detection
                   ↓
            Validate Input
                   ↓
      Deduplicate (explicit business rule)
                   ↓
          Business Key Match
                   ↓
       ┌─────────────────────┐
       │ Idempotent Write    │
       │ MERGE / Reload      │
       └─────────────────────┘
                   ↓
           Validate Target
        (row count + duplicate keys)
                   ↓
          Record SUCCESS
```

The pipeline is designed around a simple production assumption:

> **The same input may be processed more than once — and a batch-level
> check should stop it early when possible, but MERGE-based idempotency has
> to hold even when the check can't run.**

# 36. Final Takeaways

## What is idempotency?

Repeated execution should produce the same correct final state.

## Why is it important?

Production pipelines fail and are retried.

## Without idempotency, what can happen?

- Duplicate records
- Inflated revenue
- Incorrect counts
- Incorrect customer metrics
- Incorrect dashboards
- Bad downstream data
- Difficult cleanup

## What helps?

- Business keys
- Source deduplication using an explicit rule, not just `dropDuplicates()`
- Batch IDs and a real `SUCCESS → SKIP` check
- Delta `MERGE`
- Delete-and-reload — but only with a partition key that fully owns its business keys
- Target validation that checks duplicate keys, not just row counts
- Retry testing, including a real failure injected between the write and the state update

## The core Data Engineering principle

```text
Failure is possible
        ↓
Retry is possible
        ↓
Pipeline must be safe to rerun
        ↓
Idempotent pipeline
```

### The most important validation in this notebook

```text
Run 1 → 105,000 rows, 0 duplicate keys
Run 2 → 105,000 rows, 0 duplicate keys
Run 3 → 105,000 rows, 0 duplicate keys
```

The same batch was processed repeatedly, but the target did not grow with
duplicates — and, crucially, we checked for duplicate keys every time, not
just whether the row count looked stable.

---

## Next Blog

**Blog 6 — Data Quality Engineering: Building Reliable Data Pipelines**